In [1]:
# ============================================================
# CS781 Capstone - PHASE 7: Prior Case Retrieval (PCR) - Micro-F1@7
# Bi-encoder on YOUR Phase-1 LM: long case -> chunks -> chunk vectors -> document pooling -> projection ->
# unit-length case embedding. Trained contrastively (InfoNCE over in-batch + BM25 hard negatives, or margin).
# Final ranking = dense cosine, optionally fused with a from-scratch BM25 score (weight tuned on dev).
# Everything from scratch in plain PyTorch / numpy / scipy.sparse (no peft, no retrieval libraries).
#
# One notebook for all experiments - change Cell 1 only:
#   FT_METHOD: "full" | "partial" | "lora" | "qlora" | "lorma" | "adalora" | "adapter" | "prefix" | "prompt" | "ia3"
#   Knowledge distillation: set KD_TEACHER to a trained Phase-7 run's best_pcr.pt
# ============================================================

# %% 0. IMPORTS
import os, re, json, math, time, glob, random, zipfile
from collections import Counter
from types import SimpleNamespace
import numpy as np
import scipy.sparse as sp
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint
from transformers import PreTrainedTokenizerFast   # loads YOUR tokenizer file only

try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    HF_TOKEN = None

# %% 1. CONFIG  <-------------------------------------------------------- EDIT HERE
QUICK_TEST = False          # True = few queries, 1 epoch - to check the pipeline

MODEL_TYPE = "llama"        # "gpt" (Model 1) | "llama" (Model 2) | "gqa" (Model 3) | "hybrid" (Model 4)
PHASE1_CKPT = "/kaggle/input/datasets/kavyajain123987/phase1-models/best mdl2 phase1.pt"
TOKENIZER = "/kaggle/input/datasets/kavyajain123987/phase1/tokenizer (1).json"

FT_METHOD = "lora"          # full | partial | lora | qlora | lorma | adalora | adapter | prefix | prompt | ia3
FREEZE_BOTTOM = 4           # partial: freeze embeddings + this many lowest blocks
LORA_R, LORA_ALPHA = 8, 16  # lora / qlora / lorma
ADA_R_INIT, ADA_R_TARGET = 12, 4
ADAPTER_DIM = 64            # Houlsby adapters (after attention and after FFN)
N_PROMPT = 16               # prefix / prompt virtual tokens
QUANT_BLOCK = 64            # qlora 4-bit block size

# ---- case representation ----
SECTION = "spread"          # "spread" (chunks spread over the WHOLE case) | "head" | "headtail"
MAX_LEN = 512
N_CHUNKS = 4
CHUNK_POOL = "mean"         # "mean" | "last"
DOC_POOL = "mean"           # "mean" | "attn" | "max"
EMB_DIM = 256               # projection size of the case embedding

# ---- contrastive training ----
LOSS = "infonce"            # "infonce" (softmax over in-batch + hard negatives) | "margin" (pairwise hinge)
TAU = 0.05                  # InfoNCE temperature
MARGIN = 0.2                # margin loss
QUERIES_PER_STEP = 4        # each with 1 positive + HARD_NEG hard negatives; other queries' docs = in-batch negatives
HARD_NEG = 1                # hard negatives per query, sampled from BM25's top-ranked NON-relevant candidates
HARD_NEG_POOL = 30
POS_PER_QUERY = 2           # each epoch every training query is used this many times (different positives)
EPOCHS = 3
LR_BODY = 3e-5
LR_NEW = 1e-3
WEIGHT_DECAY = 0.01
WIN_BATCH = 16              # chunks per forward pass
GRAD_CKPT = False           # True = gradient checkpointing (use if CUDA out of memory)
SEED = 42

# ---- ranking ----
TOP_K = 7
FUSION = "tuned"            # "tuned" (dense + BM25, weight chosen on dev) | "dense" (embedding only) | "fixed"
BM25_WEIGHT = 0.5           # used when FUSION = "fixed"
MAX_BM25_WEIGHT = 0.9       # "tuned" never goes to pure BM25 -> the fine-tuned model always counts
BM25_K1, BM25_B = 1.2, 0.75

# ---- knowledge distillation (mandatory experiment) ----
KD_TEACHER = None           # path to a trained Phase-7 run's best_pcr.pt (file, .zip, or extracted folder)
KD_KEEP_EVERY = 2           # student = Phase-1 model with every 2nd block, fully fine-tuned
KD_T, KD_ALPHA = 1.0, 0.5   # loss = ALPHA * contrastive + (1-ALPHA) * KL(teacher score distribution || student)

OUTPUT_DIR = f"/kaggle/working/phase7_{MODEL_TYPE}_{FT_METHOD}" + ("_kd" if KD_TEACHER else "")
os.makedirs(OUTPUT_DIR, exist_ok=True)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE, torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "")

# %% 2. DOWNLOAD PHASE-7 DATA + LOOK AT IT
from huggingface_hub import snapshot_download
DATA_DIR = snapshot_download("Exploration-Lab/CS-781-Capstone", repo_type="dataset", token=HF_TOKEN,
                             allow_patterns=["Phase_7*/*", "Phase_7*/**"], local_dir="/tmp/cs781_data")
P7 = next((p for p in sorted(glob.glob(os.path.join(DATA_DIR, "Phase_7*"))) if os.path.isdir(p)), None)
assert P7, f"no Phase_7 folder in {DATA_DIR}"
FILES = sorted(glob.glob(os.path.join(P7, "**", "*.json"), recursive=True))
for f in FILES:
    print(f, f"{os.path.getsize(f) / 1e6:.1f} MB")

def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def peek(obj, n=250):
    s = json.dumps(obj, ensure_ascii=False)
    return s[:n] + ("..." if len(s) > n else "")

def find(name):
    hit = [f for f in FILES if os.path.basename(f).lower() == name]
    assert hit, f"{name} not found in {P7}"
    return hit[0]

def as_text(v):
    if isinstance(v, dict):
        v = next((v[x] for x in ("text", "document", "content") if x in v), "")
    if isinstance(v, list):
        v = " ".join(map(str, v))
    return re.sub(r"\s+", " ", str(v or "")).strip()

def parse_queries(obj):
    out = []
    items = obj.items() if isinstance(obj, dict) else ((d.get("id"), d) for d in obj)
    for k, d in items:
        rel = d.get("relevant_candidates") if isinstance(d, dict) else None
        out.append({"id": str(k), "text": as_text(d), "rel": [str(x) for x in rel] if rel is not None else None})
    return out

def parse_pool(obj):
    items = obj.items() if isinstance(obj, dict) else ((d.get("id"), d) for d in obj)
    return {str(k): as_text(d) for k, d in items}

data = {}
for split in ("train", "dev", "test"):
    q_raw, c_raw = load_json(find(f"{split}.json")), load_json(find(f"{split}_candidates.json"))
    first = q_raw[0] if isinstance(q_raw, list) else next(iter(q_raw.items()))
    print(f"\n{split}.json: {len(q_raw)} queries | first: {peek(first)}")
    queries, pool = parse_queries(q_raw), parse_pool(c_raw)
    del q_raw, c_raw
    nrel = [len(q["rel"]) for q in queries if q["rel"] is not None]
    inpool = [sum(r in pool for r in q["rel"]) for q in queries if q["rel"] is not None]
    print(f"{split}: {len(queries)} queries, {len(pool)} candidates | query words median "
          f"{int(np.median([len(q['text'].split()) for q in queries]))} | candidate words median "
          f"{int(np.median([len(t.split()) for t in list(pool.values())[:2000]]))}"
          + (f" | relevant per query mean {np.mean(nrel):.2f}, max {max(nrel)}, "
             f"{100 * sum(inpool) / max(sum(nrel), 1):.0f}% of them in the pool" if nrel else ""))
    data[split] = {"queries": queries, "pool": pool}
assert data["test"]["queries"] and data["test"]["pool"], "test not parsed - check the format above"

random.Random(SEED).shuffle(data["train"]["queries"])
if QUICK_TEST:
    data["train"]["queries"] = data["train"]["queries"][:24]
    keep = {r for q in data["train"]["queries"] for r in q["rel"]}
    others = [c for c in data["train"]["pool"] if c not in keep][:60]
    data["train"]["pool"] = {c: data["train"]["pool"][c] for c in list(keep & set(data["train"]["pool"])) + others}
    data["dev"]["queries"] = data["dev"]["queries"][:16]
    EPOCHS = 1

# %% 3. BM25 FROM SCRATCH (lexical baseline, hard-negative mining, score fusion)
WORD = re.compile(r"[a-z][a-z0-9]+|\d+")
STOP = set("the of and to in a is that for be by on as was with it this or are not an which any has have from at "
           "been his he no such shall said under were had its they all their may if so would there other but also "
           "upon who into than then them these those being can only".split())

def terms(text):
    return [w for w in WORD.findall(text.lower()) if w not in STOP]

def bm25_scores(queries, pool_ids, pool_texts):
    """(n_queries, n_candidates) BM25 matrix: sum over UNIQUE query terms of idf * saturated tf"""
    vocab, rows, cols, vals, dl = {}, [], [], [], []
    for j, t in enumerate(pool_texts):
        c = Counter(terms(t)); dl.append(sum(c.values()))
        for w, n in c.items():
            rows.append(j); cols.append(vocab.setdefault(w, len(vocab))); vals.append(n)
    tf = sp.csr_matrix((np.array(vals, dtype=np.float32), (rows, cols)), shape=(len(pool_texts), len(vocab)))
    N, dl = len(pool_texts), np.array(dl, dtype=np.float32)
    df = np.bincount(tf.indices, minlength=len(vocab))
    idf = np.log(1 + (N - df + 0.5) / (df + 0.5)).astype(np.float32)
    norm = BM25_K1 * (1 - BM25_B + BM25_B * dl / max(dl.mean(), 1))
    tf = tf.tocoo()
    w = tf.data * (BM25_K1 + 1) / (tf.data + norm[tf.row]) * idf[tf.col]
    W = sp.csr_matrix((w, (tf.row, tf.col)), shape=tf.shape)
    qr, qc = [], []
    for i, q in enumerate(queries):
        for wd in set(terms(q["text"])):
            if wd in vocab:
                qr.append(i); qc.append(vocab[wd])
    Q = sp.csr_matrix((np.ones(len(qr), dtype=np.float32), (qr, qc)), shape=(len(queries), len(vocab)))
    return np.asarray((Q @ W.T).todense(), dtype=np.float32)

t0 = time.time()
for split, D in data.items():
    D["cand_ids"] = list(D["pool"])
    D["bm25"] = bm25_scores(D["queries"], D["cand_ids"], [D["pool"][c] for c in D["cand_ids"]])
    D["cidx"] = {c: j for j, c in enumerate(D["cand_ids"])}
    for i, q in enumerate(D["queries"]):          # a case is never its own prior case
        if q["id"] in D["cidx"]:
            D["bm25"][i, D["cidx"][q["id"]]] = -1e9
    print(f"BM25 {split}: {D['bm25'].shape} in {time.time() - t0:.0f}s")

# %% 4. METRIC
def micro_f1_at_k(queries, ranked, k=TOP_K):
    tp = sum(len(set(r[:k]) & set(q["rel"])) for q, r in zip(queries, ranked))
    n_pred, n_gold = k * len(queries), sum(len(q["rel"]) for q in queries)
    p, r = tp / max(n_pred, 1), tp / max(n_gold, 1)
    return {"f1": 2 * p * r / (p + r) if p + r else 0.0, "precision": p, "recall": r}

def rank(D, S):
    top = np.argsort(-S, axis=1)[:, :TOP_K]
    return [[D["cand_ids"][j] for j in row] for row in top]

for split in ("train", "dev"):
    D = data[split]
    m = micro_f1_at_k(D["queries"], rank(D, D["bm25"]))
    print(f"BM25 only, {split}: Micro-F1@7 {m['f1']:.4f} (P {m['precision']:.4f}, R {m['recall']:.4f})")
BM25_DEV = micro_f1_at_k(data["dev"]["queries"], rank(data["dev"], data["dev"]["bm25"]))

# %% 5. TOKENIZE THE PART OF EACH CASE THE MODEL READS
tokenizer = PreTrainedTokenizerFast(tokenizer_file=TOKENIZER)
tokenizer.model_max_length = int(1e9)
BUDGET = N_CHUNKS * MAX_LEN

def chunk_ids(ids_all):
    if SECTION == "spread" and len(ids_all) > BUDGET:
        starts = np.linspace(0, len(ids_all) - MAX_LEN, N_CHUNKS).astype(int)
        return [ids_all[s:s + MAX_LEN] for s in starts]
    if SECTION == "head":
        ids = ids_all[:BUDGET]
    elif SECTION == "headtail" and len(ids_all) > BUDGET:
        ids = ids_all[:BUDGET // 2] + ids_all[-(BUDGET - BUDGET // 2):]
    else:
        ids = ids_all[:BUDGET]
    ids = ids or [0]
    return [ids[i:i + MAX_LEN] for i in range(0, len(ids), MAX_LEN)]

def tokenize_texts(texts, bs=256):
    out = []
    for b in range(0, len(texts), bs):
        part = [t if SECTION != "head" else t[:BUDGET * 8] for t in texts[b:b + bs]]
        out += [chunk_ids(x) for x in tokenizer(part, add_special_tokens=False)["input_ids"]]
    return out

t0 = time.time()
for split, D in data.items():
    D["q_chunks"] = tokenize_texts([q["text"] for q in D["queries"]])
    D["c_chunks"] = tokenize_texts([D["pool"][c] for c in D["cand_ids"]])
    for q in D["queries"]:
        q["text"] = None
    D["pool"] = None
    print(f"{split}: tokenized in {time.time() - t0:.0f}s")

# %% 6. PHASE-1 MODEL DEFINITIONS (unchanged from Phase 1)
def build_model1(cfg):
    # ---------- Model 1: GPT-style ----------
    class CausalSelfAttention(nn.Module):
        def __init__(self, d_model, n_head, dropout):
            super().__init__()
            assert d_model % n_head == 0
            self.d_model = d_model
            self.n_head = n_head
            self.head_dim = d_model // n_head
            self.c_attn = nn.Linear(d_model, 3 * d_model)
            self.c_proj = nn.Linear(d_model, d_model)
            self.attn_dropout = dropout
            self.resid_dropout = nn.Dropout(dropout)

        def forward(self, x):
            B, T, C = x.shape
            qkv = self.c_attn(x)
            q, k, v = qkv.split(self.d_model, dim=2)
            q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
            k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
            v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
            y = F.scaled_dot_product_attention(
                q, k, v, attn_mask=None,
                dropout_p=(self.attn_dropout if self.training else 0.0),
                is_causal=True)
            y = y.transpose(1, 2).contiguous().view(B, T, C)
            y = self.c_proj(y)
            y = self.resid_dropout(y)
            return y


    class MLP(nn.Module):
        def __init__(self, d_model, dropout):
            super().__init__()
            self.fc = nn.Linear(d_model, 4 * d_model)
            self.proj = nn.Linear(4 * d_model, d_model)
            self.dropout = nn.Dropout(dropout)

        def forward(self, x):
            x = self.fc(x)
            x = F.gelu(x)
            x = self.proj(x)
            x = self.dropout(x)
            return x


    class TransformerBlock(nn.Module):
        def __init__(self, d_model, n_head, dropout):
            super().__init__()
            self.ln_1 = nn.LayerNorm(d_model)
            self.attn = CausalSelfAttention(d_model, n_head, dropout)
            self.ln_2 = nn.LayerNorm(d_model)
            self.mlp = MLP(d_model, dropout)

        def forward(self, x):
            x = x + self.attn(self.ln_1(x))
            x = x + self.mlp(self.ln_2(x))
            return x


    class LegalGPT(nn.Module):
        def __init__(self, vocab_size, block_size, n_layer, n_head, d_model, dropout):
            super().__init__()
            self.block_size = block_size
            self.wte = nn.Embedding(vocab_size, d_model)
            self.wpe = nn.Embedding(block_size, d_model)
            self.blocks = nn.ModuleList([TransformerBlock(d_model, n_head, dropout) for _ in range(n_layer)])
            self.ln_f = nn.LayerNorm(d_model)
            self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
            self.lm_head.weight = self.wte.weight
            self.apply(self._init_weights)

        def _init_weights(self, module):
            if isinstance(module, nn.Linear):
                nn.init.normal_(module.weight, mean=0.0, std=0.02)
                if module.bias is not None:
                    nn.init.zeros_(module.bias)
            elif isinstance(module, nn.Embedding):
                nn.init.normal_(module.weight, mean=0.0, std=0.02)

        def forward(self, idx, targets=None):
            B, T = idx.shape
            assert T <= self.block_size
            positions = torch.arange(0, T, device=idx.device)
            x = self.wte(idx) + self.wpe(positions)
            for block in self.blocks:
                x = block(x)
            x = self.ln_f(x)
            logits = self.lm_head(x)
            loss = None
            if targets is not None:
                loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
            return logits, loss
    return LegalGPT(**cfg)


def build_model2(cfg):
    # ---------- Model 2: LLaMA-style ----------
    class RMSNorm(nn.Module):
        """x / sqrt(mean(x^2) + eps) * weight  (no mean subtraction, unlike LayerNorm)"""
        def __init__(self, dim, eps=1e-6):
            super().__init__()
            self.weight = nn.Parameter(torch.ones(dim))
            self.eps = eps

        def forward(self, x):
            rms = torch.rsqrt(x.pow(2).mean(dim=-1, keepdim=True) + self.eps)
            return (x * rms) * self.weight


    class RotaryEmbedding(nn.Module):
        """RoPE: rotate q and k by position-dependent angles (relative position info)."""
        def __init__(self, dim, max_seq_len, theta=10000.0):
            super().__init__()
            self.dim = dim
            self.max_seq_len = max_seq_len
            inv_freq = 1.0 / (theta ** (torch.arange(0, dim, 2).float() / dim))
            self.register_buffer("inv_freq", inv_freq, persistent=False)
            self._build_cache(max_seq_len)

        def _build_cache(self, seq_len):
            positions = torch.arange(seq_len, device=self.inv_freq.device, dtype=self.inv_freq.dtype)
            freqs = torch.outer(positions, self.inv_freq)
            emb = torch.cat([freqs, freqs], dim=-1)
            self.register_buffer("cos_cached", emb.cos()[None, :, :], persistent=False)
            self.register_buffer("sin_cached", emb.sin()[None, :, :], persistent=False)

        def forward(self, q, k):
            seq_len = q.size(-2)
            if seq_len > self.cos_cached.size(1):
                self._build_cache(seq_len)
            cos = self.cos_cached[:, :seq_len, :].to(q.dtype)
            sin = self.sin_cached[:, :seq_len, :].to(q.dtype)
            return apply_rotary(q, cos, sin), apply_rotary(k, cos, sin)


    def rotate_half(x):
        x1 = x[..., : x.shape[-1] // 2]
        x2 = x[..., x.shape[-1] // 2:]
        return torch.cat((-x2, x1), dim=-1)


    def apply_rotary(x, cos, sin):
        return x * cos + rotate_half(x) * sin


    class SwiGLU(nn.Module):
        """down( SiLU(gate(x)) * up(x) ), no bias"""
        def __init__(self, d_model, ffn_dim):
            super().__init__()
            self.gate_proj = nn.Linear(d_model, ffn_dim, bias=False)
            self.up_proj = nn.Linear(d_model, ffn_dim, bias=False)
            self.down_proj = nn.Linear(ffn_dim, d_model, bias=False)

        def forward(self, x):
            return self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x))


    class CausalSelfAttention(nn.Module):
        def __init__(self, d_model, n_heads, max_seq_len):
            super().__init__()
            assert d_model % n_heads == 0
            self.d_model = d_model
            self.n_heads = n_heads
            self.head_dim = d_model // n_heads
            self.qkv = nn.Linear(d_model, 3 * d_model, bias=False)
            self.out_proj = nn.Linear(d_model, d_model, bias=False)
            self.rope = RotaryEmbedding(self.head_dim, max_seq_len)

        def forward(self, x):
            B, T, C = x.shape
            q, k, v = self.qkv(x).chunk(3, dim=-1)
            q = q.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
            k = k.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
            v = v.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
            q, k = self.rope(q, k)
            y = F.scaled_dot_product_attention(q, k, v, dropout_p=0.0, is_causal=True)
            y = y.transpose(1, 2).contiguous().view(B, T, C)
            return self.out_proj(y)


    class LLaMABlock(nn.Module):
        def __init__(self, d_model, n_heads, ffn_dim, max_seq_len):
            super().__init__()
            self.attn_norm = RMSNorm(d_model)
            self.attention = CausalSelfAttention(d_model, n_heads, max_seq_len)
            self.ffn_norm = RMSNorm(d_model)
            self.ffn = SwiGLU(d_model, ffn_dim)

        def forward(self, x):
            x = x + self.attention(self.attn_norm(x))   # pre-norm
            x = x + self.ffn(self.ffn_norm(x))
            return x


    class LLaMALanguageModel(nn.Module):
        def __init__(self, cfg):
            super().__init__()
            self.cfg = cfg
            self.token_embedding = nn.Embedding(cfg.vocab_size, cfg.d_model)
            self.blocks = nn.ModuleList([LLaMABlock(cfg.d_model, cfg.n_heads, cfg.ffn_dim, cfg.max_seq_len)
                                         for _ in range(cfg.n_layers)])
            self.final_norm = RMSNorm(cfg.d_model)
            self.lm_head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
            self.lm_head.weight = self.token_embedding.weight      # weight tying
            self.apply(self._init_weights)

        def _init_weights(self, module):
            if isinstance(module, nn.Linear):
                nn.init.normal_(module.weight, mean=0.0, std=0.02)
                if module.bias is not None:
                    nn.init.zeros_(module.bias)
            elif isinstance(module, nn.Embedding):
                nn.init.normal_(module.weight, mean=0.0, std=0.02)

        def forward(self, idx, targets=None):
            x = self.token_embedding(idx)          # no position embedding: RoPE handles position
            for block in self.blocks:
                x = block(x)
            x = self.final_norm(x)
            logits = self.lm_head(x)
            loss = None
            if targets is not None:
                loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), targets.reshape(-1))
            return logits, loss
        # (your generate() method is not needed: the pipeline's batched generate_batch() is used)
    return LLaMALanguageModel(SimpleNamespace(**cfg))


def build_model3(cfg):
    # ---------- Model 3: LLaMA + GQA + sliding window ----------
    class RMSNorm(nn.Module):
        """x / sqrt(mean(x^2) + eps) * weight  (no mean subtraction, unlike LayerNorm)"""
        def __init__(self, dim, eps=1e-6):
            super().__init__()
            self.weight = nn.Parameter(torch.ones(dim))
            self.eps = eps

        def forward(self, x):
            rms = torch.rsqrt(x.pow(2).mean(dim=-1, keepdim=True) + self.eps)
            return (x * rms) * self.weight


    class RotaryEmbedding(nn.Module):
        """RoPE: rotate q and k by position-dependent angles (relative position info)."""
        def __init__(self, dim, max_seq_len, theta=10000.0):
            super().__init__()
            self.dim = dim
            self.max_seq_len = max_seq_len
            inv_freq = 1.0 / (theta ** (torch.arange(0, dim, 2).float() / dim))
            self.register_buffer("inv_freq", inv_freq, persistent=False)
            self._build_cache(max_seq_len)

        def _build_cache(self, seq_len):
            positions = torch.arange(seq_len, device=self.inv_freq.device, dtype=self.inv_freq.dtype)
            freqs = torch.outer(positions, self.inv_freq)
            emb = torch.cat([freqs, freqs], dim=-1)
            self.register_buffer("cos_cached", emb.cos()[None, :, :], persistent=False)
            self.register_buffer("sin_cached", emb.sin()[None, :, :], persistent=False)

        def forward(self, q, k):
            seq_len = q.size(-2)
            if seq_len > self.cos_cached.size(1):
                self._build_cache(seq_len)
            cos = self.cos_cached[:, :seq_len, :].to(q.dtype)
            sin = self.sin_cached[:, :seq_len, :].to(q.dtype)
            return apply_rotary(q, cos, sin), apply_rotary(k, cos, sin)


    def rotate_half(x):
        half = x.shape[-1] // 2
        x1, x2 = x[..., :half], x[..., half:]
        return torch.cat([-x2, x1], dim=-1)


    def apply_rotary(x, cos, sin):
        return x * cos + rotate_half(x) * sin


    class SwiGLU(nn.Module):
        """down( SiLU(gate(x)) * up(x) ), no bias"""
        def __init__(self, d_model, ffn_dim):
            super().__init__()
            self.gate_proj = nn.Linear(d_model, ffn_dim, bias=False)
            self.up_proj = nn.Linear(d_model, ffn_dim, bias=False)
            self.down_proj = nn.Linear(ffn_dim, d_model, bias=False)

        def forward(self, x):
            return self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x))


    class GQASlidingWindowAttention(nn.Module):
        """
        GQA: n_heads query heads share n_kv_heads key/value heads (8 Q -> 2 KV: each KV head serves 4 Q heads).
        Sliding window: token i attends only to tokens j with 0 <= i - j < window_size (causal + local).
        """
        def __init__(self, d_model, n_heads, n_kv_heads, max_seq_len, window_size):
            super().__init__()
            assert d_model % n_heads == 0
            assert n_heads % n_kv_heads == 0
            self.d_model = d_model
            self.n_heads = n_heads
            self.n_kv_heads = n_kv_heads
            self.head_dim = d_model // n_heads
            self.window_size = window_size
            self.q_proj = nn.Linear(d_model, n_heads * self.head_dim, bias=False)
            self.k_proj = nn.Linear(d_model, n_kv_heads * self.head_dim, bias=False)
            self.v_proj = nn.Linear(d_model, n_kv_heads * self.head_dim, bias=False)
            self.out_proj = nn.Linear(d_model, d_model, bias=False)
            self.rope = RotaryEmbedding(self.head_dim, max_seq_len)
            # mask[q, k] = True -> q may attend to k   (k <= q  AND  q - k < window)
            positions = torch.arange(max_seq_len)
            distance = positions[:, None] - positions[None, :]
            self.register_buffer("window_mask", (distance >= 0) & (distance < window_size), persistent=False)

        def forward(self, x):
            B, T, C = x.shape
            q = self.q_proj(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)     # [B, 8, T, D]
            k = self.k_proj(x).view(B, T, self.n_kv_heads, self.head_dim).transpose(1, 2)  # [B, 2, T, D]
            v = self.v_proj(x).view(B, T, self.n_kv_heads, self.head_dim).transpose(1, 2)  # [B, 2, T, D]
            q, k = self.rope(q, k)
            repeat_factor = self.n_heads // self.n_kv_heads
            k = k.repeat_interleave(repeat_factor, dim=1)                                  # [B, 8, T, D]
            v = v.repeat_interleave(repeat_factor, dim=1)
            mask = self.window_mask[:T, :T]
            y = F.scaled_dot_product_attention(q, k, v, attn_mask=mask, dropout_p=0.0)
            y = y.transpose(1, 2).contiguous().view(B, T, C)
            return self.out_proj(y)


    class LLaMABlock(nn.Module):
        def __init__(self, d_model, n_heads, n_kv_heads, ffn_dim, max_seq_len, window_size):
            super().__init__()
            self.attn_norm = RMSNorm(d_model)
            self.attention = GQASlidingWindowAttention(d_model=d_model, n_heads=n_heads, n_kv_heads=n_kv_heads,
                                                       max_seq_len=max_seq_len, window_size=window_size)
            self.ffn_norm = RMSNorm(d_model)
            self.ffn = SwiGLU(d_model, ffn_dim)

        def forward(self, x):
            x = x + self.attention(self.attn_norm(x))   # pre-norm
            x = x + self.ffn(self.ffn_norm(x))
            return x


    class LLaMALanguageModel(nn.Module):
        def __init__(self, cfg):
            super().__init__()
            self.cfg = cfg
            self.token_embedding = nn.Embedding(cfg.vocab_size, cfg.d_model)
            self.blocks = nn.ModuleList([
                LLaMABlock(d_model=cfg.d_model, n_heads=cfg.n_heads, n_kv_heads=cfg.n_kv_heads, ffn_dim=cfg.ffn_dim,
                           max_seq_len=cfg.max_seq_len, window_size=cfg.window_size)
                for _ in range(cfg.n_layers)])
            self.final_norm = RMSNorm(cfg.d_model)
            self.lm_head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
            self.lm_head.weight = self.token_embedding.weight      # weight tying
            self.apply(self._init_weights)

        def _init_weights(self, module):
            if isinstance(module, nn.Linear):
                nn.init.normal_(module.weight, mean=0.0, std=0.02)
                if module.bias is not None:
                    nn.init.zeros_(module.bias)
            elif isinstance(module, nn.Embedding):
                nn.init.normal_(module.weight, mean=0.0, std=0.02)

        def forward(self, idx, targets=None):
            x = self.token_embedding(idx)          # no position embedding: RoPE handles position
            for block in self.blocks:
                x = block(x)
            x = self.final_norm(x)
            logits = self.lm_head(x)
            loss = None
            if targets is not None:
                loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), targets.reshape(-1))
            return logits, loss
    return LLaMALanguageModel(SimpleNamespace(**cfg))


def build_model4(cfg):
    # ---------- Model 4: hybrid attention + gated conv ----------
    def rotate_half(x):
        x1 = x[..., : x.shape[-1] // 2]
        x2 = x[..., x.shape[-1] // 2:]
        return torch.cat((-x2, x1), dim=-1)


    class RotaryEmbedding(nn.Module):
        """RoPE: rotate q and k by position-dependent angles."""
        def __init__(self, dim, max_seq_len=2048, base=10000):
            super().__init__()
            self.dim = dim
            self.max_seq_len = max_seq_len
            self.base = base
            inv_freq = 1.0 / (base ** (torch.arange(0, dim, 2).float() / dim))
            self.register_buffer("inv_freq", inv_freq, persistent=False)
            self._build_cache(max_seq_len)

        def _build_cache(self, seq_len):
            positions = torch.arange(seq_len, dtype=self.inv_freq.dtype, device=self.inv_freq.device)
            freqs = torch.outer(positions, self.inv_freq)
            emb = torch.cat([freqs, freqs], dim=-1)
            self.register_buffer("cos_cached", emb.cos()[None, None, :, :], persistent=False)
            self.register_buffer("sin_cached", emb.sin()[None, None, :, :], persistent=False)

        def forward(self, q, k):
            seq_len = q.shape[-2]
            if seq_len > self.cos_cached.shape[-2]:
                self._build_cache(seq_len)
            # FIX: cast to q's dtype. Otherwise fp16 q * fp32 cos -> fp32 q/k while v stays fp16,
            # and scaled_dot_product_attention can fail with a dtype mismatch.
            cos = self.cos_cached[:, :, :seq_len, :].to(q.dtype)
            sin = self.sin_cached[:, :, :seq_len, :].to(q.dtype)
            q = q * cos + rotate_half(q) * sin
            k = k * cos + rotate_half(k) * sin
            return q, k


    class RMSNorm(nn.Module):
        def __init__(self, dim, eps=1e-6):
            super().__init__()
            self.weight = nn.Parameter(torch.ones(dim))
            self.eps = eps

        def forward(self, x):
            rms = torch.rsqrt(x.pow(2).mean(dim=-1, keepdim=True) + self.eps)
            return x * rms * self.weight


    class SwiGLU(nn.Module):
        def __init__(self, d_model, hidden_dim):
            super().__init__()
            self.w1 = nn.Linear(d_model, hidden_dim, bias=False)
            self.w2 = nn.Linear(hidden_dim, d_model, bias=False)
            self.w3 = nn.Linear(d_model, hidden_dim, bias=False)

        def forward(self, x):
            return self.w2(F.silu(self.w1(x)) * self.w3(x))


    class CausalSelfAttention(nn.Module):
        """GLOBAL mixer: token i attends to tokens 0..i."""
        def __init__(self, d_model, n_heads, max_seq_len):
            super().__init__()
            assert d_model % n_heads == 0
            self.d_model = d_model
            self.n_heads = n_heads
            self.head_dim = d_model // n_heads
            self.qkv = nn.Linear(d_model, 3 * d_model, bias=False)
            self.out_proj = nn.Linear(d_model, d_model, bias=False)
            self.rope = RotaryEmbedding(self.head_dim, max_seq_len=max_seq_len)

        def forward(self, x):
            B, T, C = x.shape
            q, k, v = self.qkv(x).chunk(3, dim=-1)
            q = q.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
            k = k.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
            v = v.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
            q, k = self.rope(q, k)
            y = F.scaled_dot_product_attention(q, k, v, attn_mask=None, dropout_p=0.0, is_causal=True)
            y = y.transpose(1, 2).contiguous().view(B, T, C)
            return self.out_proj(y)


    class GatedCausalConv(nn.Module):
        """
        LOCAL mixer: Linear(D->2D) -> (value, gate);
        value -> causal depthwise Conv1d (left padding only: position t sees t-k+1 .. t, never t+1);
        out = Linear( conv(value) * sigmoid(gate) ).
        """
        def __init__(self, d_model, kernel_size=5):
            super().__init__()
            self.d_model = d_model
            self.kernel_size = kernel_size
            self.in_proj = nn.Linear(d_model, 2 * d_model, bias=False)
            self.conv = nn.Conv1d(d_model, d_model, kernel_size=kernel_size, groups=d_model, bias=False)
            self.out_proj = nn.Linear(d_model, d_model, bias=False)

        def forward(self, x):
            value, gate = self.in_proj(x).chunk(2, dim=-1)
            value = value.transpose(1, 2)                          # [B, D, T] for Conv1d
            value = F.pad(value, (self.kernel_size - 1, 0))        # LEFT padding only -> causal
            value = self.conv(value).transpose(1, 2)               # back to [B, T, D]
            return self.out_proj(value * torch.sigmoid(gate))


    class HybridBlock(nn.Module):
        def __init__(self, d_model, n_heads, ffn_dim, max_seq_len, conv_kernel, use_attention):
            super().__init__()
            self.use_attention = use_attention
            self.norm1 = RMSNorm(d_model)
            self.mixer = (CausalSelfAttention(d_model=d_model, n_heads=n_heads, max_seq_len=max_seq_len)
                          if use_attention else GatedCausalConv(d_model=d_model, kernel_size=conv_kernel))
            self.norm2 = RMSNorm(d_model)
            self.ffn = SwiGLU(d_model=d_model, hidden_dim=ffn_dim)

        def forward(self, x):
            x = x + self.mixer(self.norm1(x))
            x = x + self.ffn(self.norm2(x))
            return x


    class HybridLM(nn.Module):
        def __init__(self, vocab_size, d_model, n_layers, n_heads, ffn_dim, max_seq_len, conv_kernel):
            super().__init__()
            self.vocab_size = vocab_size
            self.d_model = d_model
            self.token_embedding = nn.Embedding(vocab_size, d_model)
            # even layers -> attention, odd layers -> gated conv
            self.blocks = nn.ModuleList([
                HybridBlock(d_model=d_model, n_heads=n_heads, ffn_dim=ffn_dim, max_seq_len=max_seq_len,
                            conv_kernel=conv_kernel, use_attention=(i % 2 == 0))
                for i in range(n_layers)])
            self.final_norm = RMSNorm(d_model)
            self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
            self.lm_head.weight = self.token_embedding.weight      # weight tying
            self.apply(self._init_weights)

        def _init_weights(self, module):
            if isinstance(module, nn.Linear):
                nn.init.normal_(module.weight, mean=0.0, std=0.02)
                if module.bias is not None:
                    nn.init.zeros_(module.bias)
            elif isinstance(module, nn.Embedding):
                nn.init.normal_(module.weight, mean=0.0, std=0.02)

        def forward(self, input_ids, targets=None):
            x = self.token_embedding(input_ids)
            for block in self.blocks:
                x = block(x)
            x = self.final_norm(x)
            logits = self.lm_head(x)
            loss = None
            if targets is not None:
                # FIX: the pipeline's targets are ALREADY shifted by one (y = x[1:]).
                # Your version shifted again (logits[:, :-1] vs targets[:, 1:]) -> the model
                # would learn to predict 2 tokens ahead. Same loss as Models 1-3 now.
                loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
            return logits, loss
    return HybridLM(**cfg)


BUILDERS = {"gpt": build_model1, "llama": build_model2, "gqa": build_model3, "hybrid": build_model4}

# %% 7. ADAPTATION METHODS (from scratch)
class LoRALinear(nn.Module):
    """LoRA:   y = W x + s * B A x,  s = alpha/r.   rsLoRA: same but s = alpha/sqrt(r) (rank-stabilised)."""
    def __init__(self, base, r, alpha, rank_stabilized=False):
        super().__init__()
        self.base = base
        self.scale = alpha / math.sqrt(r) if rank_stabilized else alpha / r
        self.A = nn.Parameter(torch.randn(r, base.in_features) / math.sqrt(base.in_features))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))

    def forward(self, x):
        return self.base(x) + self.scale * F.linear(F.linear(x, self.A), self.B)


class LoRMALinear(nn.Module):
    """LoRMA (multiplicative):  y = (I + s * B A) W x.  Check the exact variant against the LoRMA paper."""
    def __init__(self, base, r, alpha):
        super().__init__()
        self.base, self.scale = base, alpha / r
        self.A = nn.Parameter(torch.randn(r, base.out_features) / math.sqrt(base.out_features))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))

    def forward(self, x):
        y = self.base(x)
        return y + self.scale * F.linear(F.linear(y, self.A), self.B)


class AdaLoRALinear(nn.Module):
    """AdaLoRA:  y = W x + s * P diag(lambda * mask) Q x  (SVD-form update).
    Ranks are pruned during training by importance |lambda * dL/dlambda| (EMA), down to a global budget."""
    def __init__(self, base, r, alpha):
        super().__init__()
        self.base, self.scale = base, alpha / r
        self.P = nn.Parameter(torch.randn(base.out_features, r) * 0.02)
        self.Q = nn.Parameter(torch.randn(r, base.in_features) * 0.02)
        self.lam = nn.Parameter(torch.zeros(r))
        self.register_buffer("mask", torch.ones(r))
        self.register_buffer("importance", torch.zeros(r))

    def forward(self, x):
        h = F.linear(x, self.Q) * (self.lam * self.mask)
        return self.base(x) + self.scale * F.linear(h, self.P)

    def orth_reg(self):
        r = self.lam.numel()
        I = torch.eye(r, device=self.P.device)
        return ((self.P.T @ self.P - I) ** 2).sum() + ((self.Q @ self.Q.T - I) ** 2).sum()


def wrap_linears(module, fn):
    """replace every nn.Linear `child` (attribute name `name`) by fn(child, name) unless it returns None"""
    for name, child in module.named_children():
        if isinstance(child, nn.Linear):
            new = fn(child, name)
            if new is not None:
                setattr(module, name, new)
        else:
            wrap_linears(child, fn)

class Quant4Linear(nn.Module):
    """Frozen weight stored in 4 bits (QLoRA-style): blockwise absmax, 2 codes packed per byte.
    (QLoRA uses the NF4 code book; this uses uniform 4-bit levels -7..7 - say so in the report.)"""
    def __init__(self, base, block=64):
        super().__init__()
        W = base.weight.data.float()
        self.out_features, self.in_features = W.shape
        flat = W.flatten()
        pad = (-flat.numel()) % block
        flat = torch.cat([flat, flat.new_zeros(pad)]).view(-1, block)
        scale = flat.abs().amax(1, keepdim=True).clamp(min=1e-8) / 7.0
        q = torch.clamp(torch.round(flat / scale), -7, 7).to(torch.int16) + 8      # 1..15
        q = q.flatten().to(torch.uint8)
        if q.numel() % 2: q = torch.cat([q, q.new_zeros(1)])
        self.register_buffer("packed", (q[0::2] | (q[1::2] << 4)).contiguous())
        self.register_buffer("scale", scale.half())
        self.numel, self.block = W.numel(), block
        self.bias = None if base.bias is None else nn.Parameter(base.bias.data.clone(), requires_grad=False)

    def weight(self):
        q = torch.stack([self.packed & 15, self.packed >> 4], 1).flatten()[: self.scale.numel() * self.block]
        W = (q.view(-1, self.block).float() - 8.0) * self.scale.float()
        return W.flatten()[: self.numel].view(self.out_features, self.in_features)

    def forward(self, x):
        return F.linear(x, self.weight().to(x.dtype), None if self.bias is None else self.bias.to(x.dtype))

class IA3Linear(nn.Module):
    """IA3: learned rescaling vector (init 1). mode="out": scales the layer output (keys/values);
    mode="in": scales the input of the FFN down-projection (= the FFN hidden activations)."""
    def __init__(self, base, mode):
        super().__init__()
        self.base, self.mode = base, mode
        self.l = nn.Parameter(torch.ones(base.out_features if mode == "out" else base.in_features))

    def forward(self, x):
        return self.base(x) * self.l if self.mode == "out" else self.base(x * self.l)

IA3_OUT = {"c_attn", "qkv", "k_proj", "v_proj"}   # attention K/V projections (fused qkv: whole output)
IA3_IN = {"proj", "down_proj", "w2"}               # FFN down-projections (their input = FFN activations)


class SubAdapter(nn.Module):
    """Bottleneck adapter on the output of one sub-layer (attention or FFN), before the residual add:
    h = sublayer(x);  h + up(GELU(down(h)))   (up initialised to 0 -> starts as the identity)"""
    def __init__(self, sublayer, d_model, dim):
        super().__init__()
        self.sublayer = sublayer
        self.down, self.up = nn.Linear(d_model, dim), nn.Linear(dim, d_model)
        nn.init.zeros_(self.up.weight); nn.init.zeros_(self.up.bias)

    def forward(self, x):
        h = self.sublayer(x)
        return h + self.up(F.gelu(self.down(h)))

ATTN_NAMES = ("attn", "attention", "mixer")    # GPT | LLaMA/GQA | hybrid (attention or gated conv)
FFN_NAMES = ("mlp", "ffn")

def add_adapters(blocks, d_model, dim, where):
    """where="houlsby": adapter after the attention AND the FFN sub-layer of every block (Houlsby 2019)
       where="pfeiffer": adapter only after the FFN sub-layer (Pfeiffer 2020, fewer parameters)"""
    for blk in blocks:
        names = (ATTN_NAMES + FFN_NAMES) if where == "houlsby" else FFN_NAMES
        for n in names:
            if hasattr(blk, n):
                setattr(blk, n, SubAdapter(getattr(blk, n), d_model, dim))

def as_ckpt_file(path):
    """Kaggle sometimes auto-extracts an uploaded .pt (it is a zip inside) into a FOLDER.
    If `path` is such a folder, re-pack it into a normal .pt file that torch.load can read."""
    if not os.path.isdir(path):
        return path
    root = next((r for r, _, f in os.walk(path) if "data.pkl" in f), None)
    assert root, f"{path} is a folder without data.pkl - not an extracted checkpoint"
    out = os.path.join("/kaggle/working", os.path.basename(path.rstrip("/")) + "_repacked.pt")
    with zipfile.ZipFile(out, "w", zipfile.ZIP_STORED) as z:
        for r, _, files in os.walk(root):
            for f in files:
                full = os.path.join(r, f)
                z.write(full, "archive/" + os.path.relpath(full, root))
    print(f"Re-packed extracted checkpoint folder {path} -> {out}")
    return out

# %% 8. BI-ENCODER: chunk encoder (+ soft prompt / prefix) -> document pooling -> projection -> unit vector
def emb_module(bb):
    return bb.wte if hasattr(bb, "wte") else bb.token_embedding

def final_norm(bb):
    return bb.ln_f if hasattr(bb, "ln_f") else bb.final_norm


class RetrModel(nn.Module):
    def __init__(self, backbone, d_model, method, n_prompt, chunk_pool, doc_pool, emb_dim):
        super().__init__()
        self.bb, self.method, self.chunk_pool, self.doc_pool = backbone, method, chunk_pool, doc_pool
        self.n_prompt = n_prompt if method in ("prompt", "prefix") else 0
        if method == "prompt":
            self.prompt = nn.Parameter(torch.randn(n_prompt, d_model) * 0.02)
        if method == "prefix":
            self.prefix = nn.Parameter(torch.randn(len(backbone.blocks) + 1, n_prompt, d_model) * 0.02)
        if doc_pool == "attn":
            self.att = nn.Sequential(nn.Linear(d_model, d_model // 2), nn.Tanh(), nn.Linear(d_model // 2, 1))
        self.proj = nn.Linear(d_model, emb_dim)

    def hidden(self, ids):
        bb, P = self.bb, self.n_prompt
        x = emb_module(bb)(ids)
        if P:
            pre = self.prompt if self.method == "prompt" else self.prefix[0]
            x = torch.cat([pre.unsqueeze(0).expand(ids.size(0), -1, -1).to(x.dtype), x], 1)
        if hasattr(bb, "wpe"):
            x = x + bb.wpe(torch.arange(x.size(1), device=ids.device))
        for i, blk in enumerate(bb.blocks):
            if self.method == "prefix" and i > 0:
                x = torch.cat([self.prefix[i].unsqueeze(0).expand(ids.size(0), -1, -1).to(x.dtype), x[:, P:]], 1)
            x = checkpoint(blk, x, use_reentrant=False) if (GRAD_CKPT and self.training) else blk(x)
        return final_norm(bb)(x)[:, P:]

    def encode(self, docs_chunks):
        """list of documents (each a list of token-id chunks) -> (n_docs, EMB_DIM) unit vectors"""
        flat = [(i, c) for i, chunks in enumerate(docs_chunks) for c in chunks]
        vecs = []
        for b in range(0, len(flat), WIN_BATCH):
            part = [c for _, c in flat[b:b + WIN_BATCH]]
            T = max(map(len, part))
            x = torch.zeros(len(part), T, dtype=torch.long)
            for i, c in enumerate(part):
                x[i, :len(c)] = torch.tensor(c)
            with autocast():
                h = self.hidden(x.to(DEVICE))
            for i, c in enumerate(part):
                vecs.append(h[i, len(c) - 1] if self.chunk_pool == "last" else h[i, :len(c)].mean(0))
        v = torch.stack(vecs).float()
        owner = torch.tensor([i for i, _ in flat], device=DEVICE)
        out = []
        for i in range(len(docs_chunks)):
            vi = v[owner == i]
            if self.doc_pool == "attn":
                vi = (torch.softmax(self.att(vi).squeeze(-1), 0).unsqueeze(-1) * vi).sum(0)
            elif self.doc_pool == "max":
                vi = vi.max(0).values
            else:
                vi = vi.mean(0)
            out.append(vi)
        return F.normalize(self.proj(torch.stack(out)), dim=-1)


def build_retr_model(model_type, p1_cfg, p1_state, method, chunk_pool, doc_pool, emb_dim, keep_every=None):
    bb = BUILDERS[model_type](p1_cfg)
    if p1_state is not None:
        bb.load_state_dict(p1_state)
    bb.lm_head = nn.Identity()
    if keep_every:
        bb.blocks = nn.ModuleList(list(bb.blocks)[::keep_every])
    d = emb_module(bb).weight.shape[1]
    m = RetrModel(bb, d, method, N_PROMPT, chunk_pool, doc_pool, emb_dim)
    if method == "full":
        return m
    for p in bb.parameters(): p.requires_grad = False
    if method == "partial":
        for blk in bb.blocks[FREEZE_BOTTOM:]:
            for p in blk.parameters(): p.requires_grad = True
        for p in final_norm(bb).parameters(): p.requires_grad = True
    elif method == "lora":
        wrap_linears(bb.blocks, lambda lin, n: LoRALinear(lin, LORA_R, LORA_ALPHA))
    elif method == "qlora":
        wrap_linears(bb.blocks, lambda lin, n: LoRALinear(Quant4Linear(lin, QUANT_BLOCK), LORA_R, LORA_ALPHA))
    elif method == "lorma":
        wrap_linears(bb.blocks, lambda lin, n: LoRMALinear(lin, LORA_R, LORA_ALPHA))
    elif method == "adalora":
        wrap_linears(bb.blocks, lambda lin, n: AdaLoRALinear(lin, ADA_R_INIT, LORA_ALPHA))
    elif method == "ia3":
        wrap_linears(bb.blocks, lambda lin, n: IA3Linear(lin, "out") if n in IA3_OUT
                     else IA3Linear(lin, "in") if n in IA3_IN else None)
    elif method == "adapter":
        add_adapters(bb.blocks, d, ADAPTER_DIM, "houlsby")
    elif method not in ("prefix", "prompt"):
        raise ValueError(method)
    return m

# %% 9. LOAD PHASE-1 WEIGHTS (+ KD teacher)
ck = torch.load(as_ckpt_file(PHASE1_CKPT), map_location="cpu")
P1_CFG = ck["config"]
print("Phase-1 config:", P1_CFG)
assert P1_CFG["vocab_size"] == len(tokenizer), "Tokenizer doesn't match this Phase-1 model"
assert MAX_LEN + (N_PROMPT if FT_METHOD in ("prompt", "prefix") else 0) <= \
    P1_CFG.get("block_size", P1_CFG.get("max_seq_len")), "MAX_LEN (+prompt) > Phase-1 context"

USE_AMP = DEVICE.type == "cuda"
AMP_DTYPE = torch.bfloat16 if USE_AMP and torch.cuda.is_bf16_supported() else torch.float16
autocast = lambda: torch.autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=USE_AMP)

RUN_CFG = dict(model_type=MODEL_TYPE, ft_method=FT_METHOD, chunk_pool=CHUNK_POOL, doc_pool=DOC_POOL, emb_dim=EMB_DIM,
               section=SECTION, n_chunks=N_CHUNKS, max_len=MAX_LEN, loss=LOSS, lora_r=LORA_R, lora_alpha=LORA_ALPHA,
               ada_r_init=ADA_R_INIT, adapter_dim=ADAPTER_DIM, n_prompt=N_PROMPT, freeze_bottom=FREEZE_BOTTOM,
               keep_every=KD_KEEP_EVERY if KD_TEACHER else None, phase1_cfg=P1_CFG)

teacher = None
if KD_TEACHER:
    t = torch.load(as_ckpt_file(KD_TEACHER), map_location="cpu", weights_only=False)
    tc = t["run_cfg"]
    if (tc["section"], tc["n_chunks"], tc["max_len"]) != (SECTION, N_CHUNKS, MAX_LEN):
        print("WARNING: teacher was trained on a different part of the case (SECTION / N_CHUNKS / MAX_LEN)")
    for k in ("lora_r", "lora_alpha", "ada_r_init", "adapter_dim", "n_prompt", "freeze_bottom"):
        globals()[k.upper()] = tc[k]
    teacher = build_retr_model(tc["model_type"], tc["phase1_cfg"], None, tc["ft_method"], tc["chunk_pool"],
                               tc["doc_pool"], tc["emb_dim"], tc.get("keep_every"))
    teacher.load_state_dict(t["state"])
    teacher = teacher.to(DEVICE).eval()
    for p in teacher.parameters(): p.requires_grad = False
    n_teacher = sum(p.numel() for p in teacher.parameters()) + \
        sum(q.numel for q in teacher.modules() if isinstance(q, Quant4Linear))
    print(f"KD teacher: {tc['model_type']} + {tc['ft_method']} ({n_teacher:,} params)")
    FT_METHOD = RUN_CFG["ft_method"] = "full"
    del t

model = build_retr_model(MODEL_TYPE, P1_CFG, ck["model"], FT_METHOD, CHUNK_POOL, DOC_POOL, EMB_DIM,
                         KD_KEEP_EVERY if KD_TEACHER else None).to(DEVICE)
del ck
n_total = sum(p.numel() for p in model.parameters()) + sum(q.numel for q in model.modules() if isinstance(q, Quant4Linear))
n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"{FT_METHOD}{' (KD student)' if KD_TEACHER else ''}: trainable {n_train:,} / {n_total:,} "
      f"({100 * n_train / n_total:.2f}%) | section={SECTION} {N_CHUNKS}x{MAX_LEN} | pool {CHUNK_POOL}/{DOC_POOL} | loss={LOSS}")
if KD_TEACHER:
    print(f"KD: student {n_total:,} params = {100 * n_total / n_teacher:.0f}% of teacher")

# %% 10. EVALUATION (dense, BM25, fusion)
@torch.no_grad()
def embed_all(m, chunks_list, bs=32):
    m.eval()
    return torch.cat([m.encode(chunks_list[b:b + bs]) for b in range(0, len(chunks_list), bs)]).cpu().numpy()

def zscore(S):
    S = np.where(S < -1e8, np.nan, S)
    mu, sd = np.nanmean(S, 1, keepdims=True), np.nanstd(S, 1, keepdims=True) + 1e-6
    return np.nan_to_num((S - mu) / sd, nan=-1e9)

def dense_scores(m, D):
    Q, C = embed_all(m, D["q_chunks"]), embed_all(m, D["c_chunks"])
    S = Q @ C.T
    for i, q in enumerate(D["queries"]):
        if q["id"] in D["cidx"]:
            S[i, D["cidx"][q["id"]]] = -1e9
    return S

def fuse(Sd, Sb, w):
    return (1 - w) * zscore(Sd) + w * zscore(Sb)

# %% 11. CONTRASTIVE TRAINING (in-batch + BM25 hard negatives)
TR = data["train"]
train_q = [i for i, q in enumerate(TR["queries"]) if any(r in TR["cidx"] for r in q["rel"])]
print(f"{len(train_q)} training queries with at least one relevant case in the pool")
hard = {}
for i in train_q:
    rel = {TR["cidx"][r] for r in TR["queries"][i]["rel"] if r in TR["cidx"]}
    top = np.argsort(-TR["bm25"][i])[:HARD_NEG_POOL + len(rel) + 1]
    hard[i] = [j for j in top if j not in rel and TR["bm25"][i, j] > -1e8][:HARD_NEG_POOL]

def contrastive_loss(qv, cv, owner_pos, relmask):
    """qv (B,E), cv (M,E); owner_pos[b] = column of query b's positive; relmask (B,M) True = relevant to b"""
    S = qv @ cv.T
    other_rel = relmask.clone(); other_rel[torch.arange(len(qv)), owner_pos] = False
    if LOSS == "infonce":
        return F.cross_entropy((S / TAU).masked_fill(other_rel, -1e4), owner_pos), S, other_rel
    pos = S[torch.arange(len(qv)), owner_pos].unsqueeze(1)
    neg_mask = ~relmask
    hinge = F.relu(MARGIN - pos + S) * neg_mask
    return hinge.sum() / neg_mask.sum().clamp(min=1), S, other_rel

new_ids = {id(p) for n, p in model.named_parameters()
           if not n.startswith("bb.") or n.endswith((".A", ".B", ".P", ".Q", ".lam", ".l")) or ".down." in n or ".up." in n}
groups = [{"params": [p for p in model.parameters() if p.requires_grad and id(p) in new_ids], "lr": LR_NEW},
          {"params": [p for p in model.parameters() if p.requires_grad and id(p) not in new_ids], "lr": LR_BODY}]
optimizer = torch.optim.AdamW([g for g in groups if g["params"]], weight_decay=WEIGHT_DECAY)
steps_per_epoch = math.ceil(len(train_q) * POS_PER_QUERY / QUERIES_PER_STEP)
total_steps = EPOCHS * steps_per_epoch
warmup = max(1, int(0.06 * total_steps))
sched = torch.optim.lr_scheduler.LambdaLR(
    optimizer, lambda s: min(1.0, (s + 1) / warmup) * max(0.0, (total_steps - s) / max(1, total_steps - warmup)))
scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP and AMP_DTYPE == torch.float16)

ada_mods = [m_ for m_ in model.modules() if isinstance(m_, AdaLoRALinear)]
ADA_BUDGET0 = sum(m_.lam.numel() for m_ in ada_mods)
ADA_TARGET = ADA_R_TARGET * len(ada_mods)
ADA_T0, ADA_T1, ADA_EVERY, ADA_BETA = int(0.1 * total_steps), int(0.7 * total_steps), 10, 0.85
def adalora_update(step):
    for m_ in ada_mods:
        if m_.lam.grad is not None:
            m_.importance.mul_(ADA_BETA).add_((1 - ADA_BETA) * (m_.lam * m_.lam.grad).abs().detach())
    if step < ADA_T0 or step % ADA_EVERY: return
    prog = min(1.0, (step - ADA_T0) / max(1, ADA_T1 - ADA_T0))
    budget = int(ADA_TARGET + (ADA_BUDGET0 - ADA_TARGET) * (1 - prog) ** 3)
    allimp = torch.cat([m_.importance for m_ in ada_mods])
    if budget < allimp.numel():
        thr = torch.topk(allimp, budget).values[-1]
        for m_ in ada_mods:
            m_.mask.copy_((m_.importance >= thr).float())

DV = data["dev"]
BEST_PATH = os.path.join(OUTPUT_DIR, "best_pcr.pt")
best_f1, history, step, t0 = -1.0, [], 0, time.time()
if DEVICE.type == "cuda": torch.cuda.reset_peak_memory_stats()

def evaluate_and_maybe_save(tag):
    global best_f1
    Sd = dense_scores(model, DV)
    m_d = micro_f1_at_k(DV["queries"], rank(DV, Sd))
    m_f = micro_f1_at_k(DV["queries"], rank(DV, fuse(Sd, DV["bm25"], 0.5)))
    history.append({"at": tag, "dense": m_d, "fused_0.5": m_f})
    print(f"  [{tag}] dev Micro-F1@7 dense {m_d['f1']:.4f} | dense+BM25(0.5) {m_f['f1']:.4f} | "
          f"BM25 {BM25_DEV['f1']:.4f} | {time.time() - t0:.0f}s")
    if m_d["f1"] > best_f1:
        best_f1 = m_d["f1"]
        torch.save({"state": model.state_dict(), "run_cfg": RUN_CFG}, BEST_PATH)
        print("  >> best model saved")
    model.train()

evaluate_and_maybe_save("before training")
for epoch in range(1, EPOCHS + 1):
    model.train()
    order = [i for i in train_q for _ in range(POS_PER_QUERY)]
    random.shuffle(order)
    tot, n = 0.0, 0
    for b in range(0, len(order), QUERIES_PER_STEP):
        qs = list(dict.fromkeys(order[b:b + QUERIES_PER_STEP]))          # no duplicate query in one step
        cols, owner_pos = [], []
        for i in qs:
            rel = [TR["cidx"][r] for r in TR["queries"][i]["rel"] if r in TR["cidx"]]
            p = random.choice(rel)
            if p not in cols: cols.append(p)
            owner_pos.append(cols.index(p))
            for j in random.sample(hard[i], min(HARD_NEG, len(hard[i]))):
                if j not in cols: cols.append(j)
        relmask = torch.tensor([[c in {TR["cidx"][r] for r in TR["queries"][i]["rel"] if r in TR["cidx"]}
                                 for c in cols] for i in qs], device=DEVICE)
        owner_pos = torch.tensor(owner_pos, device=DEVICE)
        qv = model.encode([TR["q_chunks"][i] for i in qs])
        cv = model.encode([TR["c_chunks"][j] for j in cols])
        loss, S, other_rel = contrastive_loss(qv, cv, owner_pos, relmask)
        if teacher is not None:
            with torch.no_grad():
                St = teacher.encode([TR["q_chunks"][i] for i in qs]) @ teacher.encode([TR["c_chunks"][j] for j in cols]).T
            # distil the teacher's ranking distribution over this step's candidates
            p_t = F.softmax((St / TAU / KD_T).masked_fill(other_rel, -1e4), -1)
            log_s = F.log_softmax((S / TAU / KD_T).masked_fill(other_rel, -1e4), -1)
            kd = F.kl_div(log_s, p_t, reduction="batchmean")
            loss = KD_ALPHA * loss + (1 - KD_ALPHA) * KD_T ** 2 * kd
        if ada_mods:
            loss = loss + 0.1 * sum(m_.orth_reg() for m_ in ada_mods) / len(ada_mods)
        optimizer.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        if ada_mods: adalora_update(step)
        torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.grad is not None], 1.0)
        scaler.step(optimizer); scaler.update(); sched.step()
        step += 1
        tot += loss.item(); n += 1
        if step % 25 == 0:
            rate = (time.time() - t0) / step
            print(f"epoch {epoch} step {step}/{total_steps} | loss {tot / n:.4f} | "
                  f"ETA training {rate * (total_steps - step) / 60:.0f} min")
    evaluate_and_maybe_save(f"epoch {epoch}")

TRAIN_MIN = (time.time() - t0) / 60
PEAK_GB = torch.cuda.max_memory_allocated() / 1024**3 if DEVICE.type == "cuda" else None
print(f"\nBest dev dense Micro-F1@7 {best_f1:.4f} | {TRAIN_MIN:.1f} min | peak GPU {PEAK_GB} GB")

# %% 12. FINAL DEV: choose the BM25 fusion weight
ckpt = torch.load(BEST_PATH, map_location=DEVICE, weights_only=False)
model.load_state_dict(ckpt["state"])
Sd = dense_scores(model, DV)
W_GRID = [round(float(w), 2) for w in np.arange(0, MAX_BM25_WEIGHT + 1e-9, 0.1)]
fusion_curve = {w: micro_f1_at_k(DV["queries"], rank(DV, fuse(Sd, DV["bm25"], w)))["f1"] for w in W_GRID}
if FUSION == "dense":
    W_BM25 = 0.0
elif FUSION == "fixed":
    W_BM25 = BM25_WEIGHT
else:
    W_BM25 = max(W_GRID, key=lambda w: (round(fusion_curve[w], 6), -w))
DEV_DENSE = micro_f1_at_k(DV["queries"], rank(DV, Sd))
DEV_FINAL = micro_f1_at_k(DV["queries"], rank(DV, fuse(Sd, DV["bm25"], W_BM25)))
print("dev Micro-F1@7 by BM25 weight:", {w: round(v, 4) for w, v in fusion_curve.items()})
print(f"FINAL dev: dense only {DEV_DENSE['f1']:.4f} | BM25 only {BM25_DEV['f1']:.4f} | "
      f"used (BM25 weight {W_BM25}) {DEV_FINAL['f1']:.4f} (P {DEV_FINAL['precision']:.4f}, R {DEV_FINAL['recall']:.4f})")
torch.save({**ckpt, "bm25_weight": W_BM25}, BEST_PATH)

# %% 13. TEST RANKING + prediction.json + ZIP
TE = data["test"]
t1 = time.time()
St = fuse(dense_scores(model, TE), TE["bm25"], W_BM25)
INFER_MS = (time.time() - t1) / (len(TE["queries"]) + len(TE["cand_ids"])) * 1000
ranked = rank(TE, St)
preds = {q["id"]: r for q, r in zip(TE["queries"], ranked)}
pool_ids = set(TE["cand_ids"])
assert len(preds) == len(TE["queries"])
assert all(len(r) == TOP_K and len(set(r)) == TOP_K and set(r) <= pool_ids and q not in r for q, r in preds.items())
print(f"{len(preds)} test queries ranked over {len(pool_ids)} candidates | {INFER_MS:.0f} ms per case embedded")
print("example:", next(iter(preds.items())))

pred_path = os.path.join(OUTPUT_DIR, "prediction.json")
with open(pred_path, "w", encoding="utf-8") as f:
    json.dump(preds, f)
zip_path = os.path.join(OUTPUT_DIR, "submission.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    z.write(pred_path, arcname="prediction.json")
print("Submission ready:", zip_path)

# %% 14. RUN SUMMARY (experiment log / report)
with open(os.path.join(OUTPUT_DIR, "run_summary.json"), "w") as f:
    json.dump({**{k: v for k, v in RUN_CFG.items() if k != "phase1_cfg"}, "kd_teacher": KD_TEACHER,
               "fusion": FUSION, "bm25_weight": W_BM25, "trainable_params": n_train, "total_params": n_total,
               "teacher_params": n_teacher if KD_TEACHER else None, "train_queries": len(train_q), "epochs": EPOCHS,
               "dev_dense": DEV_DENSE, "dev_bm25": BM25_DEV, "dev_final": DEV_FINAL,
               "dev_f1_by_bm25_weight": fusion_curve, "history": history, "train_minutes": round(TRAIN_MIN, 1),
               "peak_gpu_gb": PEAK_GB, "inference_ms_per_case": round(INFER_MS, 1)}, f, indent=2)
print("Done.")

Device: cuda Tesla T4


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

/tmp/cs781_data/Phase_7_PCR/Dev/dev.json 5.9 MB
/tmp/cs781_data/Phase_7_PCR/Dev/dev_candidates.json 51.9 MB
/tmp/cs781_data/Phase_7_PCR/Test/test.json 9.3 MB
/tmp/cs781_data/Phase_7_PCR/Test/test_candidates.json 71.1 MB
/tmp/cs781_data/Phase_7_PCR/Train/train.json 33.2 MB
/tmp/cs781_data/Phase_7_PCR/Train/train_candidates.json 159.8 MB

train.json: 827 queries | first: ["0000000599", {"text": ["PETITIONER: <NAME> Vs. RESPONDENT: ROOP SINGH RATHORE & OTHERS(and connected appeal) DATE OF JUDGMENT: 07/05/1963 BENCH: <NAME>, S.K. BENCH: <NAME>, S.K. SINHA, BHUVNESHWAR P.(CJ) DAYAL, RAGHUBAR AYYANGAR, <NAME> MUDHOLKAR, ...
train: 827 queries, 4320 candidates | query words median 4508 | candidate words median 3654 | relevant per query mean 6.42, max 52, 100% of them in the pool

dev.json: 118 queries | first: ["0000012637", {"text": ["PETITIONER: <ORG>, <NAME> Vs. RESPONDENT: COMMISSIONER OF INCOME-TAX, <NAME> DATE OF JUDGMENT: 08/12/1965 BENCH: <NAME>, S.M. BENCH: <NAME>, S.M. SUBBARAO, K. 